# Attributes, Views, and Indexing

This notebook covers the parts of the API that make `AnnNet` pleasant to inspect:

- attribute upserts for vertices, edges, slices, and graph-level metadata
- read-only dataframe-style views
- index helpers for mapping IDs to incidence rows and columns
- `GraphView` for filtered subgraph exploration


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PACKAGE_ROOT = (NOTEBOOK_DIR / "..").resolve()
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

import annnet as an


In [2]:
from annnet.core._Views import GraphView

G = an.AnnNet(directed=True)
G.add_vertices(["EGFR", "GRB2", "SOS1", "RAS"])
G.add_edge("EGFR", "GRB2", edge_id="e1")
G.add_edge("GRB2", "SOS1", edge_id="e2")
G.add_edge("SOS1", "RAS", edge_id="e3")

G.set_graph_attribute("dataset", "toy receptor signaling")
G.set_vertex_attrs("EGFR", family="RTK", compartment="membrane")
G.set_vertex_attrs("GRB2", family="adapter", compartment="cytosol")
G.set_edge_attrs("e1", relation="recruits", evidence="literature")
G.set_edge_attrs("e2", relation="binds")
G.set_edge_attrs("e3", relation="activates")


In [3]:
print("graph attributes:", G.get_graph_attributes())
print("EGFR attrs:", G.get_vertex_attrs("EGFR"))
print("e1 attrs:", G.get_edge_attrs("e1"))


graph attributes: {'dataset': 'toy receptor signaling'}
EGFR attrs: {'vertex_id': 'EGFR', 'family': 'RTK', 'compartment': 'membrane'}
e1 attrs: {'edge_id': 'e1', 'relation': 'recruits', 'evidence': 'literature'}


## Read-only dataframe views

These helpers expose the current attribute state without changing the graph.


In [4]:
G.vertices_view()


vertex_id,family,compartment
str,str,str
"""EGFR""","""RTK""","""membrane"""
"""GRB2""","""adapter""","""cytosol"""
"""SOS1""",null,null
"""RAS""",null,null


In [5]:
G.edges_view()


edge_id,kind,directed,global_weight,source,target,edge_type,head,tail,members,relation,evidence,effective_weight
str,str,bool,f64,str,str,str,list[str],list[str],list[str],str,str,f64
"""e1""","""binary""",true,1.0,"""EGFR""","""GRB2""","""regular""",null,null,null,"""recruits""","""literature""",1.0
"""e2""","""binary""",true,1.0,"""GRB2""","""SOS1""","""regular""",null,null,null,"""binds""",null,1.0
"""e3""","""binary""",true,1.0,"""SOS1""","""RAS""","""regular""",null,null,null,"""activates""",null,1.0


## Index helpers

`G.idx` is the public namespace for row and column lookups in the incidence matrix.


In [6]:
print("EGFR row:", G.idx.entity_to_row("EGFR"))
print("row 0 ->", G.idx.row_to_entity(0))
print("e2 column:", G.idx.edge_to_col("e2"))
print("index stats:", G.idx.stats())


EGFR row: 0
row 0 -> EGFR
e2 column: 1
index stats: {'n_entities': 4, 'n_vertices': 4, 'n_edge_entities': 0, 'n_edges': 3, 'max_row': 3, 'max_col': 2}


## Filtered graph views

`GraphView` is the lazy filtered view object. It does not duplicate the graph unless you explicitly materialize it.


In [7]:
signal_view = GraphView(G, vertices=["EGFR", "GRB2", "SOS1"])

print("vertex ids:", sorted(signal_view.vertex_ids))
print("edge ids:", signal_view.edge_ids)
signal_view.obs


vertex ids: ['EGFR', 'GRB2', 'SOS1']
edge ids: None


vertex_id,family,compartment
str,str,str
"""EGFR""","""RTK""","""membrane"""
"""GRB2""","""adapter""","""cytosol"""
"""SOS1""",null,null


In [8]:
materialized = signal_view.materialize()
print(materialized.shape)
materialized.edges_view()


(3, 2)


edge_id,kind,directed,global_weight,source,target,edge_type,head,tail,members,relation,evidence,effective_weight
str,str,bool,f64,str,str,str,list[str],list[str],list[str],str,str,f64
"""edge_0""","""binary""",true,1.0,"""EGFR""","""GRB2""","""regular""",null,null,null,"""recruits""","""literature""",1.0
"""edge_1""","""binary""",true,1.0,"""GRB2""","""SOS1""","""regular""",null,null,null,"""binds""",null,1.0
